# Supertonic Style Extraction

Turn any recording into a repo-compatible style JSON: `style_ttl [1, 50, 256]`
and `style_dp [1, 8, 16]`, the only two conditioning surfaces the frozen ONNX
graph exposes. Everything else in this fork -- emotion deltas, the presentation
axis, future attribute axes -- starts from styles produced here.

Run it on a GPU runtime (Runtime > Change runtime type > T4 or better).

**Two different runtimes, on purpose.** Extraction converts the frozen ONNX
graph to PyTorch with `onnx2torch` and backpropagates into the style tensor
under a WavLM perceptual loss: that needs CUDA. Playback uses `onnxruntime` on
**CPU** and must stay there -- `py/helper.py` raises
`NotImplementedError("GPU mode is not fully tested")` if asked for a GPU
session. The two halves of this notebook do not share a device.

Budget roughly 15 minutes per clip on a good GPU. Colab free tier disconnects
after about 90 minutes idle and caps a session near 12 hours, so every long
cell here checkpoints to Drive per clip and resumes by skipping finished work.

Only process recordings you have permission to process, and carry the source
corpus's licence terms into whatever you derive from it.

## References

- [supertonic.embed](https://github.com/kdrkdrkdr/supertonic.embed) -- the
  extractor this notebook drives.
- [supertonic3-voice-clone](https://github.com/saurabhv749/supertonic3-voice-clone)
  -- fallback extractor, explicitly Supertonic 3, roughly 2.6 GB peak.
- [docs/EMOTION_CALIBRATION.md](EMOTION_CALIBRATION.md) -- the prose recipe for
  emotion calibration, including the reference-recording rules this notebook
  cites rather than restates.
- [docs/emotion_calibration_colab.ipynb](emotion_calibration_colab.ipynb) --
  the sibling notebook: one speaker, matched neutral/angry/surprised, emotion
  deltas. This notebook is its general-purpose half.
- [Supertone/supertonic-3](https://huggingface.co/Supertone/supertonic-3) --
  the ONNX assets and the ten released voice presets.

## Read this before the batch run: which model version?

`supertonic.embed`'s own `models/README.md` points at
**`Supertone/supertonic-2`** for weights. This repo runs
**`Supertone/supertonic-3`**. Our README describes v3 as shipping
v2-compatible public ONNX assets, and the tensor shapes do line up
(`style_ttl` 50x256, `style_dp` 8x16) -- but no source we have states v3
compatibility outright. It is an assumption, not a verified fact.

So this notebook extracts **one clip first**, validates the result, and only
then offers the batch cell. Do not skip the smoke test: the failure mode it
guards against is a four-hour batch that produces plausible-looking JSON files
conditioned on the wrong graph.

If the smoke test fails, the smoke-test cell prints the options: point the
config at v2 assets, or switch to the `supertonic3-voice-clone` fallback.

In [ ]:
# 1. GPU check. Fail here rather than 20 minutes into an extraction.
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No CUDA device. Runtime > Change runtime type > Hardware accelerator: '
        'GPU (T4 or better), then Runtime > Run all. Style extraction '
        'backpropagates through a converted ONNX graph with a WavLM loss and '
        'is not practical on CPU.')

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print('GPU: %s' % name)
print('VRAM: %.1f GB' % vram)
print('torch: %s, CUDA: %s' % (torch.__version__, torch.version.cuda))
if vram < 6:
    print('WARNING: under 6 GB. Set TIMBRE_BATCH = 1 below (about 2.5 GB).')

In [ ]:
# 2. Drive: recordings in, styles out, model cache alongside. Nothing that
# takes minutes to produce should live only in the Colab VM.
import os
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/supertonic-style-extraction')
RECORDINGS = WORKSPACE / 'recordings'
STYLES = WORKSPACE / 'extracted-styles'
LOGS = WORKSPACE / 'logs'
for directory in (WORKSPACE, RECORDINGS, STYLES, LOGS):
    directory.mkdir(parents=True, exist_ok=True)

# WavLM-large is about 1.2 GB and downloads on first extraction. Keeping the
# HF cache on Drive means a disconnect costs minutes, not the download again.
os.environ['HF_HOME'] = str(WORKSPACE / 'huggingface-cache')

MANIFEST = WORKSPACE / 'extraction_manifest.json'
print('Workspace:   %s' % WORKSPACE)
print('Recordings:  %s' % RECORDINGS)
print('Styles out:  %s' % STYLES)
print('HF_HOME:     %s' % os.environ['HF_HOME'])

In [ ]:
# 3. Extractor, its dependencies, and a checkout of the runtime for playback.
EXTRACTOR = Path('/content/supertonic.embed')
REPO = Path('/content/supertonic')

# Set this to your fork's clone URL to get Style.with_deltas and the repo's own
# tooling. Upstream is enough for plain playback.
REPO_URL = 'https://github.com/supertone-inc/supertonic.git'

if not EXTRACTOR.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kdrkdrkdr/supertonic.embed.git',
                    str(EXTRACTOR)], check=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
                   check=True)

# Extractor requirements: torch, numpy, scipy, librosa, transformers, onnx,
# onnx2torch, onnxslim, httpx, PyYAML.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(EXTRACTOR / 'requirements.txt')], check=True)
# Inference side, pinned to py/requirements.txt in this repo.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnxruntime==1.23.1', 'numpy>=1.26.0', 'soundfile>=0.12.1',
                'librosa>=0.10.0', 'PyYAML>=6.0', 'huggingface_hub'],
               check=True)

print('Extractor: %s' % EXTRACTOR)
print('Runtime repo: %s' % REPO)

In [ ]:
# The shared module. Notebook A writes it; docs/presentation_axis_colab.ipynb
# imports it from the same Drive location, so extraction, validation and the
# blending mirror are written once.
import sys
from pathlib import Path

MODULE_SOURCE = r'''
"""Shared helpers for the Supertonic style-extraction Colab notebooks.

Written to Drive by docs/style_extraction_colab.ipynb and imported by
docs/presentation_axis_colab.ipynb, so the extraction, validation and blending
logic exists once rather than twice.

Dependency-light on purpose: numpy plus the standard library at import time.
PyYAML is imported inside the one function that needs it.
"""

import json
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

TTL_SHAPE = (50, 256)
DP_SHAPE = (8, 16)

# Rows of style_ttl that carry voice identity in the ten released presets.
# Measured by the Phase 0 row-locality probe (py/phase0_row_locality.py; see
# new-plan.md). They are scattered, not contiguous: never group by index range.
ACTIVE_ROWS = [0, 2, 5, 6, 7, 8, 9, 13, 15, 16, 18, 19, 20, 22, 23, 27,
               31, 32, 38, 42, 45, 47, 48, 49]


def patch_extractor_config(src_config, out_config, onnx_dir, presets_dir,
                           batch=None, total_step=None, seed=None):
    """Copy the extractor's config.yaml with our asset paths substituted.

    Only the keys we own are touched; every other key keeps the extractor's
    own default, and relative paths in it still resolve because the extractor
    is always run with cwd set to its repository root.
    """
    import yaml

    config = yaml.safe_load(Path(src_config).read_text())
    config.setdefault("model", {})
    config["model"]["onnx_dir"] = str(Path(onnx_dir).resolve())
    config["model"]["presets"] = str(Path(presets_dir).resolve())
    if total_step is not None:
        config["model"]["total_step"] = total_step
    if seed is not None:
        config["model"]["seed"] = seed
    if batch is not None:
        config.setdefault("timbre", {})
        config["timbre"]["batch"] = batch
    out_config = Path(out_config)
    out_config.parent.mkdir(parents=True, exist_ok=True)
    out_config.write_text(yaml.safe_dump(config, sort_keys=False))
    return out_config


def run_extraction(clip_path, name, out_dir, extractor_dir, config_path=None,
                   log_path=None):
    """Extract one clip into ``out_dir/<name>.json`` and return that path.

    Output is streamed line by line with an elapsed-seconds prefix so a long
    run visibly makes progress, and optionally tee'd to ``log_path``.
    Raises on a non-zero exit or a missing output file: an extractor that
    "succeeds" without writing a style is a failure, not a warning.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    list_path = out_dir / ("_list_" + name + ".txt")
    list_path.write_text(str(Path(clip_path).resolve()) + "|" + name + "\n")

    cmd = [sys.executable, "src/run_extract_style_batch.py", str(list_path),
           "--out", str(out_dir)]
    if config_path is not None:
        cmd += ["--config", str(Path(config_path).resolve())]

    produced = out_dir / (name + ".json")
    started = time.time()
    print("[extract] " + name + " <- " + str(clip_path), flush=True)
    log = open(str(log_path), "a") if log_path else None
    proc = subprocess.Popen(cmd, cwd=str(extractor_dir),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            stamp = "[%7.1fs] " % (time.time() - started)
            text = stamp + line.rstrip()
            print(text, flush=True)
            if log is not None:
                log.write(text + "\n")
                log.flush()
        code = proc.wait()
    finally:
        if log is not None:
            log.close()

    if code != 0:
        raise RuntimeError("Extractor exited with code %d for %r" % (code, name))
    if not produced.exists():
        seen = sorted(p.name for p in out_dir.glob("*.json"))
        raise FileNotFoundError(
            "Extractor finished but %s does not exist. JSON files in %s: %s"
            % (produced, out_dir, seen))
    print("[done] %s in %.1f min" % (name, (time.time() - started) / 60.0),
          flush=True)
    return produced


def load_style_arrays(path):
    """Load a style JSON the way py/helper.py load_voice_style does.

    ``dims`` is authoritative and ``data`` is flattened before the reshape, so
    however the emitter nested the numbers does not matter. Returns
    ``(style_ttl, style_dp)`` as float32 arrays of shape (50, 256) / (8, 16).
    """
    payload = json.loads(Path(path).read_text())
    arrays = []
    for key in ("style_ttl", "style_dp"):
        if key not in payload:
            raise KeyError("%s has no %r block" % (path, key))
        block = payload[key]
        if "dims" not in block or "data" not in block:
            raise KeyError("%s: %r needs both 'dims' and 'data'" % (path, key))
        dims = list(block["dims"])
        if len(dims) != 3:
            raise ValueError("%s: %r dims %r is not 3-D" % (path, key, dims))
        flat = np.array(block["data"], dtype=np.float32).flatten()
        expected = int(np.prod(dims))
        if flat.size != expected:
            raise ValueError("%s: %r has %d values but dims %r imply %d"
                             % (path, key, flat.size, dims, expected))
        arrays.append(flat.reshape(dims[1], dims[2]))
    return arrays[0], arrays[1]


def row_norms(ttl):
    """Per-row L2 norms of style_ttl, over the last axis."""
    return np.linalg.norm(np.asarray(ttl, dtype=np.float32), axis=-1)


def validate_style_file(path, tol=0.02, verbose=True):
    """Check one emitted style JSON. Returns a report dict; never raises for a
    merely suspicious file, but sets report["ok"] False and lists problems."""
    path = Path(path)
    problems = []
    ttl, dp = load_style_arrays(path)
    if ttl.shape != TTL_SHAPE:
        problems.append("style_ttl shape %r, expected %r" % (ttl.shape, TTL_SHAPE))
    if dp.shape != DP_SHAPE:
        problems.append("style_dp shape %r, expected %r" % (dp.shape, DP_SHAPE))
    if not np.isfinite(ttl).all():
        problems.append("style_ttl contains non-finite values")
    if not np.isfinite(dp).all():
        problems.append("style_dp contains non-finite values")

    norms = row_norms(ttl)
    if norms.size and (abs(norms - 1.0).max() > tol):
        problems.append(
            "style_ttl rows are not unit-norm: min %.4f max %.4f (tolerance %.3f). "
            "The blending API restores each row to its own norm, so a non-unit "
            "file still blends, but every measurement in this project assumes "
            "unit rows -- find out why before using it."
            % (float(norms.min()), float(norms.max()), tol))

    report = {
        "path": str(path),
        "ok": not problems,
        "ttl_shape": list(ttl.shape),
        "dp_shape": list(dp.shape),
        "ttl_row_norm_min": float(norms.min()) if norms.size else None,
        "ttl_row_norm_max": float(norms.max()) if norms.size else None,
        "ttl_row_norm_mean": float(norms.mean()) if norms.size else None,
        "dp_row_norm_min": float(np.linalg.norm(dp, axis=-1).min()),
        "dp_row_norm_max": float(np.linalg.norm(dp, axis=-1).max()),
        "problems": problems,
    }
    if verbose:
        status = "OK  " if report["ok"] else "FAIL"
        print("[%s] %s  ttl%s dp%s  ttl row norms %.4f..%.4f"
              % (status, path.name, tuple(ttl.shape), tuple(dp.shape),
                 report["ttl_row_norm_min"], report["ttl_row_norm_max"]))
        for problem in problems:
            print("       ! " + problem)
    return report


def write_style_json(path, ttl, dp, metadata=None):
    """Write tensors in the repo's style JSON schema (batch dim included)."""
    ttl = np.asarray(ttl, dtype=np.float32)
    dp = np.asarray(dp, dtype=np.float32)
    payload = {
        "style_ttl": {"data": [ttl.tolist()],
                      "dims": [1, int(ttl.shape[0]), int(ttl.shape[1])],
                      "type": "float32"},
        "style_dp": {"data": [dp.tolist()],
                     "dims": [1, int(dp.shape[0]), int(dp.shape[1])],
                     "type": "float32"},
    }
    if metadata is not None:
        payload["metadata"] = metadata
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload))
    return path


def _restore_row_norms(blended, reference):
    target = np.linalg.norm(reference, axis=-1, keepdims=True)
    current = np.linalg.norm(blended, axis=-1, keepdims=True).clip(min=1e-8)
    return (blended / current * target).astype(blended.dtype, copy=False)


def with_deltas_np(ttl, dp, deltas, include_duration=False):
    """Numpy mirror of Style.with_deltas in py/helper.py.

    Accumulates every (delta_ttl, delta_dp, weight) in pre-normalization space
    and restores per-row norms exactly once, over the last axis. Zero weight is
    an exact identity: the normalize is never round-tripped.

    This exists so the notebook can preview an axis without depending on the
    fork being cloned. It is a mirror, not a second implementation to maintain:
    the artifacts these notebooks emit are consumed by py/helper.py and
    web/helper.js, which remain the two implementations of record. The
    notebooks cross-check against py/helper.py whenever the fork is available.
    """
    ttl = np.asarray(ttl, dtype=np.float32).copy()
    dp = np.asarray(dp, dtype=np.float32).copy()
    active = [(d_ttl, d_dp, float(w)) for d_ttl, d_dp, w in deltas
              if float(w) != 0.0]
    if not active:
        return ttl, dp
    base_ttl = ttl.copy()
    out_ttl = ttl.copy()
    for d_ttl, _, w in active:
        out_ttl = out_ttl + np.float32(w) * np.asarray(d_ttl, dtype=np.float32)
    out_ttl = _restore_row_norms(out_ttl, base_ttl)
    if include_duration:
        base_dp = dp.copy()
        out_dp = dp.copy()
        for _, d_dp, w in active:
            out_dp = out_dp + np.float32(w) * np.asarray(d_dp, dtype=np.float32)
        out_dp = _restore_row_norms(out_dp, base_dp)
    else:
        out_dp = dp
    return out_ttl, out_dp


def format_eta(done, total, elapsed_s):
    """'3/16 done, 41.2 min elapsed, ~2.8 h remaining' -- for long batches."""
    if done <= 0:
        return "%d/%d done, %.1f min elapsed, remaining unknown" % (
            done, total, elapsed_s / 60.0)
    per = elapsed_s / done
    remaining = per * (total - done)
    unit = "min" if remaining < 5400 else "h"
    value = remaining / 60.0 if unit == "min" else remaining / 3600.0
    return "%d/%d done, %.1f min elapsed, ~%.1f %s remaining" % (
        done, total, elapsed_s / 60.0, value, unit)
'''

module_path = WORKSPACE / 'style_tools.py'
module_path.write_text(MODULE_SOURCE.strip() + '\n')
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

import style_tools
import importlib
importlib.reload(style_tools)

print('Wrote ' + str(module_path))
print('Active style_ttl rows (%d): %s'
      % (len(style_tools.ACTIVE_ROWS), style_tools.ACTIVE_ROWS))

In [ ]:
# 4. Model assets. snapshot_download, not git-lfs: 392 MB over lfs is flaky in
# Colab, and the resume logic here is the whole point.
from huggingface_hub import snapshot_download

ASSETS = Path('/content/supertonic-assets')
snapshot_download(
    repo_id='Supertone/supertonic-3',
    local_dir=str(ASSETS),
    allow_patterns=['onnx/*', 'voice_styles/*'],
)

ONNX_DIR = ASSETS / 'onnx'
PRESET_DIR = ASSETS / 'voice_styles'
missing = [name for name in ('duration_predictor.onnx', 'text_encoder.onnx',
                             'vector_estimator.onnx', 'vocoder.onnx',
                             'tts.json', 'unicode_indexer.json')
           if not (ONNX_DIR / name).exists()]
if missing:
    raise FileNotFoundError('Missing ONNX assets: %s' % missing)

print('ONNX: %s' % sorted(p.name for p in ONNX_DIR.iterdir()))
print('Presets: %s' % sorted(p.stem for p in PRESET_DIR.glob('*.json')))

In [ ]:
# 5. Extractor config. We only override the asset paths (and optionally the
# batch size); every other key keeps the extractor's own default. The patched
# copy lives on Drive, so re-cloning the extractor does not lose it.
TIMBRE_BATCH = None   # None keeps the extractor default (8). Use 1 for ~2.5 GB.

CONFIG = style_tools.patch_extractor_config(
    src_config=EXTRACTOR / 'src' / 'config.yaml',
    out_config=WORKSPACE / 'extractor_config.yaml',
    onnx_dir=ONNX_DIR,
    presets_dir=PRESET_DIR,
    batch=TIMBRE_BATCH,
)
print(CONFIG.read_text())

## What to feed it

The rules this project already committed to, in
[docs/EMOTION_CALIBRATION.md](EMOTION_CALIBRATION.md) -- follow them there
rather than inventing new ones here:

- **One style per recording.** Never train a single style from several
  recordings you want to distinguish later. A delta is a difference between two
  separately extracted styles, not a mixture.
- **Matched content for matched conditions.** For emotion, the same speaker,
  the same or closely matched text, one emotion per file. Anything that varies
  besides the thing you are measuring ends up inside the delta.
- **One speaker is a pipeline test, not a general vector.** At least five
  speakers for a prototype, 8-20 for a general vector, and subtract each
  speaker's own neutral before averaging across speakers.
- **Licensing travels with the recordings.** RAVDESS is CC BY-NC-SA 4.0; keep
  attribution and the non-commercial/share-alike terms on anything derived from
  it. Check the terms of any other corpus at its source.

Practical points for this notebook specifically:

- Cost is roughly 15 minutes **per clip**, near enough independent of clip
  length, so short is not cheaper -- but a few seconds of clean speech is what
  the extractor is designed for. Prefer a clean, single-speaker, non-clipped
  recording with little background noise or reverb.
- Name files for what they are (`actor01_angry.wav`, `p225_003_mic1.wav`): the
  file stem becomes the style name and the output JSON filename.
- Mixed formats are fine as inputs, but the optional conversion below writes
  mono WAV at the source sample rate, which removes any format question from
  the extraction step.

In [ ]:
# 6. Collect the clips. Either drop files into the Drive recordings folder
# (survives disconnects, preferred) or upload them into this session.
INPUT_MODE = 'drive'          # 'drive' or 'upload'
CONVERT_TO_MONO_WAV = True    # decode to mono WAV at the source sample rate
AUDIO_SUFFIXES = ('.wav', '.flac', '.mp3', '.ogg', '.m4a')

import shutil

if INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    for filename in uploaded:
        shutil.copy2(filename, RECORDINGS / filename)
elif INPUT_MODE != 'drive':
    raise ValueError("INPUT_MODE must be 'drive' or 'upload'")

found = sorted(p for p in RECORDINGS.iterdir()
               if p.suffix.lower() in AUDIO_SUFFIXES and not p.name.startswith('.'))
if not found:
    raise FileNotFoundError(
        'No audio in %s. Copy files there in the Drive UI, or set '
        "INPUT_MODE = 'upload' and rerun this cell." % RECORDINGS)

if CONVERT_TO_MONO_WAV:
    import librosa
    import soundfile as sf
    prepared_dir = WORKSPACE / 'prepared'
    prepared_dir.mkdir(exist_ok=True)
    prepared = []
    for path in found:
        target = prepared_dir / (path.stem + '.wav')
        if not target.exists():
            audio, sr = librosa.load(str(path), sr=None, mono=True)
            sf.write(str(target), audio, sr)
        prepared.append(target)
    found = prepared

CLIPS = []
seen = {}
for path in found:
    name = ''.join(ch if (ch.isalnum() or ch in '-_') else '_' for ch in path.stem)
    if name in seen:
        raise ValueError('Two inputs map to the style name %r: %s and %s. '
                         'Rename one.' % (name, seen[name], path))
    seen[name] = path
    CLIPS.append((path, name))

import soundfile as sf
print('%-28s %-10s %s' % ('style name', 'duration', 'path'))
for path, name in CLIPS:
    try:
        info = sf.info(str(path))
        duration = '%.2fs' % (info.frames / float(info.samplerate))
    except Exception as exc:              # unreadable header, still extractable
        duration = '?'
    print('%-28s %-10s %s' % (name, duration, path))
print('\n%d clip(s), roughly %.1f h at 15 min/clip.'
      % (len(CLIPS), len(CLIPS) * 0.25))

## Smoke test: one clip, then look at it

This is the cell that decides whether the v2/v3 assumption holds. It extracts
the first clip only, loads the result exactly the way `py/helper.py`
`load_voice_style` does -- `dims` is authoritative, `data` is flattened, cast
to float32 -- and checks shape, finiteness and per-row norms.

`style_ttl` rows are expected to be unit-norm over the last axis. That is a
project-wide invariant, not a cosmetic detail: every delta, every axis fit and
the whole blending API assume it. A file whose rows are not unit-norm is a
finding, not something to average past.

In [ ]:
# 7. Smoke test: extract exactly one clip and validate before committing hours.
import json
import traceback

smoke_path, smoke_name = CLIPS[0]
smoke_dir = WORKSPACE / 'smoke-test'
smoke_dir.mkdir(exist_ok=True)
smoke_json = smoke_dir / (smoke_name + '.json')

if smoke_json.exists():
    print('Smoke test already extracted: %s (delete it to redo)' % smoke_json)
else:
    try:
        style_tools.run_extraction(
            clip_path=smoke_path, name=smoke_name, out_dir=smoke_dir,
            extractor_dir=EXTRACTOR, config_path=CONFIG,
            log_path=LOGS / 'smoke_test.log')
    except Exception:
        traceback.print_exc()
        print('\n' + '=' * 72)
        print('SMOKE TEST FAILED -- do not run the batch cell.')
        print('=' * 72)
        print(
            'Known open question: supertonic.embed documents Supertone/'
            'supertonic-2 weights, and this notebook pointed it at\n'
            'supertonic-3. The shapes match and this repo treats the v3 ONNX\n'
            'assets as v2-compatible, but that compatibility is not stated by\n'
            'any source we have.\n\n'
            'Options, in order:\n'
            '  1. Read the traceback. A shape or node-name mismatch inside\n'
            '     onnx2torch is the version signature; an OOM or a missing\n'
            '     file is not.\n'
            '  2. Re-run the asset cell with repo_id=\'Supertone/supertonic-2\'\n'
            '     into a separate directory and re-point CONFIG at it. Styles\n'
            '     extracted against v2 must then be checked against the v3\n'
            '     runtime by ear in the playback cell before you trust them.\n'
            '  3. Fall back to github.com/saurabhv749/supertonic3-voice-clone,\n'
            '     which targets Supertonic 3 explicitly:\n'
            '       python train_style.py --name my-voice \\\n'
            '         --target-wav-path voices/my-voice.wav \\\n'
            '         --num-steps 3000 --learning-rate 0.0002\n'
            '     Peak memory is about 2.6 GB. Its output still has to satisfy\n'
            '     the validation cell below before anything else uses it.\n'
            '  4. Set TIMBRE_BATCH = 1 and retry if the failure is memory.')
        raise

report = style_tools.validate_style_file(smoke_json, verbose=True)
print(json.dumps(report, indent=2))
if not report['ok']:
    raise RuntimeError(
        'The extractor produced a file, but it does not look like a usable '
        'style (see problems above). Treat this exactly like a hard failure: '
        'the batch would produce more of the same.')
# Promote the smoke-test result into the real output directory so the batch
# cell skips this clip instead of paying another 15 minutes for it.
import shutil
promoted = STYLES / smoke_json.name
if not promoted.exists():
    shutil.copy2(smoke_json, promoted)
    print('Copied smoke-test style to %s' % promoted)

print('\nSmoke test passed. The extractor and the v3 assets agree on shapes '
      'and the rows are unit-norm. Proceed to the batch cell.')

In [ ]:
# 8. Batch extraction. Per-clip checkpoint to Drive, resume by skipping any
# clip whose JSON already validates, one failure does not sink the run.
import time
from datetime import datetime, timezone

manifest = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}
started_at = time.time()
completed = 0
pending = []

for path, name in CLIPS:
    target = STYLES / (name + '.json')
    if target.exists():
        try:
            if style_tools.validate_style_file(target, verbose=False)['ok']:
                manifest[name] = {'status': 'complete', 'path': str(target),
                                  'source': str(path)}
                print('[skip] %s: checkpoint present and valid' % name)
                continue
            print('[redo] %s: checkpoint present but invalid, re-extracting' % name)
            target.unlink()
        except Exception as exc:
            print('[redo] %s: checkpoint unreadable (%s), re-extracting' % (name, exc))
            target.unlink()
    pending.append((path, name))

print('\n%d to extract, %d already done. Estimated %.1f h.\n'
      % (len(pending), len(CLIPS) - len(pending), len(pending) * 0.25))

for path, name in pending:
    manifest[name] = {'status': 'running', 'source': str(path),
                      'started_at': datetime.now(timezone.utc).isoformat()}
    MANIFEST.write_text(json.dumps(manifest, indent=2))
    try:
        produced = style_tools.run_extraction(
            clip_path=path, name=name, out_dir=STYLES, extractor_dir=EXTRACTOR,
            config_path=CONFIG, log_path=LOGS / ('extract_' + name + '.log'))
        report = style_tools.validate_style_file(produced, verbose=True)
        manifest[name] = {
            'status': 'complete' if report['ok'] else 'suspect',
            'path': str(produced), 'source': str(path),
            'completed_at': datetime.now(timezone.utc).isoformat(),
            'validation': report,
        }
    except Exception as exc:
        manifest[name] = {'status': 'failed', 'source': str(path),
                          'error': repr(exc),
                          'failed_at': datetime.now(timezone.utc).isoformat()}
        print('[FAILED] %s: %r' % (name, exc))
    MANIFEST.write_text(json.dumps(manifest, indent=2))
    completed += 1
    print('[progress] ' + style_tools.format_eta(
        completed, len(pending), time.time() - started_at) + '\n')

states = {}
for name, entry in manifest.items():
    states.setdefault(entry['status'], []).append(name)
print('Manifest: %s' % MANIFEST)
for status in sorted(states):
    print('  %-9s %s' % (status, sorted(states[status])))

In [ ]:
# 9. Validation sweep over everything in the output directory, loaded exactly
# the way py/helper.py would load it. This is the gate before anything
# downstream (deltas, axes, listening sets) touches these files.
emitted = sorted(p for p in STYLES.glob('*.json')
                 if p.name != 'extraction_report.json')
if not emitted:
    raise FileNotFoundError('No style JSON files in %s' % STYLES)

reports = []
for path in emitted:
    try:
        reports.append(style_tools.validate_style_file(path, verbose=True))
    except Exception as exc:
        print('[FAIL] %s: %r' % (path.name, exc))
        reports.append({'path': str(path), 'ok': False, 'problems': [repr(exc)]})

good = [r for r in reports if r.get('ok')]
bad = [r for r in reports if not r.get('ok')]
print('\n%d/%d valid.' % (len(good), len(reports)))
if good:
    lows = [r['ttl_row_norm_min'] for r in good]
    highs = [r['ttl_row_norm_max'] for r in good]
    print('style_ttl per-row norms across all valid files: %.5f .. %.5f'
          % (min(lows), max(highs)))
    print('(Expected ~1.0. The whole project -- deltas, axes, with_deltas() '
          'row-norm restoration -- assumes unit rows.)')
if bad:
    print('\nPROBLEM FILES:')
    for r in bad:
        print('  %s' % r['path'])
        for problem in r.get('problems', []):
            print('    ! %s' % problem)
    print('\nDo not build deltas or axes on these.')

(WORKSPACE / 'validation_report.json').write_text(json.dumps(reports, indent=2))
print('\nWrote %s' % (WORKSPACE / 'validation_report.json'))

## Hear it (CPU)

Playback is `onnxruntime` on CPU and must stay there: `py/helper.py` raises
`NotImplementedError("GPU mode is not fully tested")` for a GPU session. The
GPU is idle for this cell, which is fine -- eight denoising steps of a short
sentence on a Colab CPU is seconds, not minutes.

The vocoder's initial latent is drawn with an unseeded `np.random.randn`, so
the same style rendered twice is audibly different. numpy is re-seeded
immediately before each synthesis so clips are comparable to each other; any
"is this identical" check still belongs at the tensor level, not on audio.

Post these clips into the conversation, not just to disk -- a style is judged
by ear first.

In [ ]:
# 10. Synthesize one sentence per extracted style, on CPU.
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

sys.path.insert(0, str(REPO / 'py'))
import helper

for attr in ('Style', 'load_text_to_speech', 'load_voice_style'):
    if not hasattr(helper, attr):
        raise AttributeError(
            'py/helper.py in %s has no %r. REPO_URL points at a checkout that '
            'does not match this fork; set it to your fork and rerun the clone '
            'cell.' % (REPO, attr))

# Same sentence and settings as py/phase0_linearity_gate.py, so these clips are
# directly comparable with the project's existing listening sets.
TEXT = 'The quick brown fox jumps over the lazy dog.'
LANG = 'en'
TOTAL_STEP = 8
SPEED = 1.05
SEED = 0
MAX_PREVIEWS = 6

text_to_speech = helper.load_text_to_speech(str(ONNX_DIR))   # CPU by default
preview_dir = WORKSPACE / 'previews'
preview_dir.mkdir(exist_ok=True)

playable = [Path(r['path']) for r in reports if r.get('ok')][:MAX_PREVIEWS]
if len(good) > MAX_PREVIEWS:
    print('Previewing the first %d of %d styles.\n' % (MAX_PREVIEWS, len(good)))

for path in playable:
    style = helper.load_voice_style([str(path)])
    np.random.seed(SEED)
    wav, duration = text_to_speech(TEXT, LANG, style, TOTAL_STEP, SPEED)
    trimmed = wav[0, :int(text_to_speech.sample_rate * duration[0].item())]
    out_path = preview_dir / (path.stem + '.wav')
    sf.write(str(out_path), trimmed, text_to_speech.sample_rate)
    print('%s | style=%s | text=%r | total_step=%d speed=%.2f seed=%d lang=%s'
          % (out_path.name, path.stem, TEXT, TOTAL_STEP, SPEED, SEED, LANG))
    display(Audio(str(out_path)))

In [ ]:
# 11. Take the results with you. The Drive copy is the durable one; this is for
# getting styles onto the machine that runs the repo.
import shutil

from google.colab import files

bundle = Path('/content/supertonic-styles')
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
for path in emitted:
    shutil.copy2(path, bundle / path.name)
if MANIFEST.exists():
    shutil.copy2(MANIFEST, bundle / MANIFEST.name)
shutil.copy2(WORKSPACE / 'validation_report.json', bundle / 'validation_report.json')

archive = shutil.make_archive('/content/supertonic-styles', 'zip', str(bundle))
print('Archive: %s (%.1f MB)' % (archive, Path(archive).stat().st_size / 1e6))
print('Drive copy: %s' % STYLES)
print('\nExtracted styles, recordings and generated WAVs are git-ignored in '
      'this repo. Copy them into assets/, do not commit them.')
files.download(archive)